# FastAPI – Practice Exercises

Hands-on exercises for FastAPI, following `g_fw-fastapi.ipynb`.

Use `from fastapi.testclient import TestClient` to test without a running server.

### Contents

- [Exercise 1 – Hello World App](#exercise-1--hello-world-app)
- [Exercise 2 – Path, Query, and Body Parameters](#exercise-2--path-query-and-body-parameters)
- [Exercise 3 – Pydantic Request and Response Models](#exercise-3--pydantic-request-and-response-models)
- [Exercise 4 – Dependency Injection](#exercise-4--dependency-injection)
- [Exercise 5 – Background Tasks](#exercise-5--background-tasks)
- [Exercise 6 – Error Handling and Custom Responses](#exercise-6--error-handling-and-custom-responses)
- [Exercise 7 – Testing with TestClient](#exercise-7--testing-with-testclient)
- [Exercise 8 – Async Routes and External Services](#exercise-8--async-routes-and-external-services)
- [Exercise 9 – Dependency Overrides for Testing](#exercise-9--dependency-overrides-for-testing)
- [Exercise 10 – Project Structure and Best Practices](#exercise-10--project-structure-and-best-practices)

## Exercise 1 – Hello World App

**Goal**: Create a minimal FastAPI application.

**Requirements**:

- Instantiate `app = FastAPI()`.
- Add a `GET /` route that returns `{"message": "hello"}`.
- Use `TestClient(app)` to call the route and assert the response.

**Hint**: `TestClient` wraps `requests` and runs the app in-process.

## Exercise 2 – Path, Query, and Body Parameters

**Goal**: Accept different kinds of input in a single endpoint.

**Requirements**:

- Add `GET /symbols/{ticker}` that returns the ticker as JSON.
- Add a `limit: int = 10` query parameter.
- Add `POST /orders` that accepts a JSON body with `symbol`, `qty`, `price`.

**Hint**: Path params are declared as function arguments matching the `{name}` in the route.

## Exercise 3 – Pydantic Request and Response Models

**Goal**: Validate and document request/response shapes.

**Requirements**:

- Define `OrderIn(BaseModel)` with `symbol: str`, `qty: int`, `price: float`.
- Define `OrderOut(OrderIn)` adding `id: int` and `created_at: datetime`.
- Use `response_model=OrderOut` on the POST route; return a fake `OrderOut` object.

**Hint**: `response_model` filters the output to only the declared fields.

## Exercise 4 – Dependency Injection

**Goal**: Share reusable dependencies across routes.

**Requirements**:

- Write a `get_db()` generator that yields a fake in-memory dict.
- Inject it into a route with `db: dict = Depends(get_db)`.
- Write a `get_current_user()` dependency that reads a `token` query param and raises `HTTPException(401)` if missing.

**Hint**: `Depends(fn)` calls `fn` for each request and injects the returned value.

## Exercise 5 – Background Tasks

**Goal**: Run work after the response is sent.

**Requirements**:

- Add a `BackgroundTasks` parameter to a POST route.
- Call `background_tasks.add_task(send_alert, order_id)` where `send_alert` logs a message.
- Verify the route returns immediately and the task runs afterwards.

**Hint**: `BackgroundTasks` is built-in; for heavy work prefer a real task queue like Celery.

## Exercise 6 – Error Handling and Custom Responses

**Goal**: Return meaningful error responses.

**Requirements**:

- Raise `HTTPException(status_code=404, detail='Symbol not found')` for an unknown ticker.
- Register an exception handler for `ValueError` that returns a 422 response with a message.
- Test both the 404 and 422 paths with `TestClient`.

**Hint**: `@app.exception_handler(ExcType)` registers a global handler for a given exception class.

## Exercise 7 – Testing with TestClient

**Goal**: Write a full test suite for a small trading API.

**Requirements**:

- Build a `/orders` CRUD API (list, create, get by id) backed by an in-memory list.
- Write tests for: create order (201), get existing (200), get missing (404), list all (200).
- Assert response JSON fields and the length of the list after several creates.

**Hint**: Reset shared state between tests using a `setup_function` or dependency override.

## Exercise 8 – Async Routes and External Services

**Goal**: Make async requests from an async route.

**Requirements**:

- Declare an `async def` route that uses `httpx.AsyncClient` to fetch a URL.
- Use `asyncio.gather` inside the route to fetch two URLs concurrently.
- In tests, mock `httpx.AsyncClient.get` to avoid real network calls.

**Hint**: `httpx` has an `AsyncClient` that integrates well with FastAPI's async test helpers.

## Exercise 9 – Dependency Overrides for Testing

**Goal**: Swap real dependencies for test doubles.

**Requirements**:

- Define a `get_settings()` dependency that reads `os.environ`.
- In a test, use `app.dependency_overrides[get_settings] = lambda: FakeSettings(...)` to inject fixed values.
- Verify the route uses the overridden settings.

**Hint**: `app.dependency_overrides` is a plain dict; clear it in teardown to avoid test pollution.

## Exercise 10 – Project Structure and Best Practices

**Goal**: Organise a production-ready FastAPI project.

**Requirements**:

- Print the recommended directory layout: `app/main.py`, `app/routers/`, `app/models/`, `app/dependencies.py`.
- Explain how `APIRouter` keeps routes modular and how `app.include_router(router, prefix='/v1')` assembles them.
- List three FastAPI-specific best practices.

**Hint**: `APIRouter` is to FastAPI what Blueprint is to Flask.